In [0]:
%run ../../config/utils

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import lit
import pyspark.sql.functions as f
import pyspark.sql
import pandas as pd
import os
import datetime
from ast import literal_eval

In [0]:
run_date_str = dbutils.widgets.get('run_as_date')
muhh_path = dbutils.widgets.get('muhh_path')
train_flag = literal_eval(dbutils.widgets.get('train'))
current_year = int(run_date_str[:4])
past_year = current_year - 1

if train_flag:
    filtering_date_str = f"{past_year}-08-31"
    filtering_date = pd.to_datetime(filtering_date_str)
    dataset_cd = 'train'
else:
    filtering_date_str = run_date_str
    filtering_date = pd.to_datetime(filtering_date_str)
    dataset_cd = 'inference'

# Prepare base dataframe

In [0]:
if train_flag:
    #Previous year assignment file
    last_score_date_py = spark.table(gm_scores).filter(f.col('score_date')<=f'{past_year}-12-31').select('score_date').agg(f.max('score_date').alias('max_score_dt')).first()['max_score_dt']
    Assignment_file = spark.table(gm_holdout_and_mailer).filter((f.col('score_date') == last_score_date_py) & (f.col('output_type') == 'Holdout')).drop('output_type').toPandas()
    GM_Score_2024 = spark.table(gm_scores).filter(f.col('score_date') == last_score_date_py).toPandas()

    deciles_7_8_9_10 = GM_Score_2024[GM_Score_2024['decile'].isin([7,8,9,10])][['MBRSHP_SID','decile']].sample(n=80_000, random_state=0)

    data = pd.concat([Assignment_file[['MBRSHP_SID','decile']], deciles_7_8_9_10[['MBRSHP_SID','decile']]]).drop_duplicates().reset_index(drop=True)
    Assignment_file_sp = spark.createDataFrame(data[['MBRSHP_SID']])
    # data.groupby(['decile']).agg({'MBRSHP_SID':'nunique'})

    # Customers how purchased GM Category
    data_header = spark.table(silver_transaction_fiscal_header).filter(f.col('SALES_CHANNEL_ID').isin([10,30])).select('PURCH_HDR_ID')

    data_detail = spark.table(silver_transaction_fiscal_detail).filter(f.col('PURCH_DT').between(f'{past_year}-11-01',f'{past_year}-12-31')).filter(f.col('RETURN_IND')=='N')

    data = data_header.join(data_detail,['PURCH_HDR_ID'],'inner')

    is_GM = f.col('MCH3_CD').like('4%')
    mark_GM = f.when(is_GM, f.lit(1)).otherwise(0)
    data = data.withColumn('is_GM', mark_GM)
    transaction_columns = ['mbrshp_sid','is_GM']
    data = data.select(transaction_columns).dropDuplicates()
    data = data.groupby('mbrshp_sid').agg(f.sum('is_GM').alias('GM_purchase'))
    # data.count()
    # data.groupby('GM_purchase').agg(sqlf.countDistinct('mbrshp_sid')).show()

    # Merge with last year holdout group
    Assignment_file_sp_v2 = Assignment_file_sp.withColumnRenamed("MBRSHP_SID","mbrshp_sid").join(data, 'mbrshp_sid', "left").fillna(
    0, 
    subset = ['GM_purchase'],)
    # Assignment_file_sp_v2.groupby('GM_purchase').agg(sqlf.countDistinct('mbrshp_sid')).show()
    mbr_dna = spark.table(fs_customer_cube_full).filter(f.col('FISCAL_WEEK_END') == filtering_date_str).filter(f.col('MBRSHP_STAT_CD')=='AM').filter(f.col('LATEST_MBRSHP_EXP_DT')>filtering_date - pd.DateOffset(days=95))

    base = Assignment_file_sp_v2.join(mbr_dna,on='mbrshp_sid', how='inner')#.fillna(0, subset=['GM_purchase'])
    # print(base.count())
    # base.groupby('GM_purchase').agg(sqlf.countDistinct('mbrshp_sid')).show()
    # base.groupby(['GM_purchase','TENURE_GROUP']).agg(sqlf.countDistinct('mbrshp_sid')).show()
    base = base.withColumnRenamed('mbrshp_sid', 'MBRSHP_SID')
else:
    max_fiscal_week_end = spark.table(fs_customer_cube_full).select(f.max('FISCAL_WEEK_END').alias('max_dt')).first()['max_dt']
    base = spark.table(fs_customer_cube_full).filter(f.col('FISCAL_WEEK_END') == max_fiscal_week_end)#.filter(f.col('MBRSHP_STAT_CD')=='AM').filter(f.col('LATEST_MBRSHP_EXP_DT')>filtering_date-pd.DateOffset(days=95))#.filter(f.col('LATEST_MBRSHP_SUB_TYPE')=='P')
    if muhh_path:
        MUHH = spark.read.csv(muhh_path,header=True)
        MUHH = MUHH.withColumn('MBRSHP_SID',f.col('MBRSHP_SID').cast('long'))
        base = MUHH.select('MBRSHP_SID').distinct().join(base,['MBRSHP_SID'],how='inner')
    # base.groupby(['TENURE_GROUP']).agg(f.countDistinct('MBRSHP_SID')).show()


## Adding additional columns for member DNA

In [0]:
#Member Frequency group
low_frequency_visits_last_26_weeks_cutoff=0
high_frequency_visits_last_12_weeks_cutoff=12

base = (
    base.withColumn(
        'MEMBER_FREQUENCY_GROUP',
        f.when(
            base["LAST_TWENTY-SIX_WEEK_TRIPS"]
            <= low_frequency_visits_last_26_weeks_cutoff,
            "LOW",
        )
        .when(
            base["LAST_TWELVE_WEEK_TRIPS"]
            >= high_frequency_visits_last_12_weeks_cutoff,
            "HIGH",
        )
        .otherwise("MEDIUM"),
    )
)

#Shopped in last 12 weeks
base = (
    base.withColumn(
        'Shopped_in_Last_3Month',
        f.when(
            f.col('LAST_TWELVE_WEEK_TRIPS') > 0,
            1
        )
        .otherwise(0)
    )
)

# base

## Adding additional columns with respect to transaction

In [0]:
purch_header = spark.table(silver_transaction_fiscal_header).filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095), filtering_date)).filter(f.col('SALES_CHANNEL_ID').isin([10, 30, 40])).select('PURCH_HDR_ID','SALES_CHANNEL_ID') #TODO Revisar fechas con Hazim

purch_detail = spark.table(silver_transaction_fiscal_detail).filter(f.col('SALES_CTGRY_CD') == '03').filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095), filtering_date)).filter(f.col('RETURN_IND')=='N').select('MBRSHP_SID','PURCH_DT','PURCH_HDR_ID','DISCOUNT_TYPE_CD','ARTICLE_NBR','NORMAL_PRC_AMT','SALES_CTGRY_CD','MCH3_CD','MCH2_CD','MCH2_DESC','MCH1_CD','AH3_DESC','AH4_DESC')

# purch_detail.select(min('purch_dt'), max('purch_dt')).show()

In [0]:
"""
purch_detail_raw = (
    purch_detail_raw.withColumnRenamed(
        'purch_hdr_id', 
        'PURCH_HDR_ID'
    )    
    .withColumnRenamed(
        'sales_channel_id', 
        'SALES_CHANNEL_ID'
    )
    .drop_duplicates()
)
"""

purch_detail = (
    purch_detail.join(
        purch_header,
        on='PURCH_HDR_ID',
        how='left'
    )
)

In [0]:
# Sundries trips
# 1 year
customers_non_edible_last_1y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '300000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Sundries_trips_last_1y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_sundries_sub_cat_last_1y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Sundries_sales_last_1y'),
        
    )
)


customers_non_edible_last_2y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '300000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=720),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Sundries_trips_last_2y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_sundries_sub_cat_last_2y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Sundries_sales_last_2y'),
        
    )
)


customers_non_edible_last_3y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '300000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095),filtering_date_str))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Sundries_trips_last_3y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_sundries_sub_cat_last_3y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '300000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Sundries_sales_last_3y'),
        
    )
)

Sundries = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        customers_non_edible_last_1y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_non_edible_last_2y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_non_edible_last_3y,
        on=['MBRSHP_SID'],
        how='left'
    )
    .fillna(0, subset=['Sundries_trips_last_1y','Sundries_trips_last_2y','Sundries_trips_last_3y','unique_sundries_sub_cat_last_1y','unique_sundries_sub_cat_last_2y','unique_sundries_sub_cat_last_3y','Sundries_sales_last_1y','Sundries_sales_last_2y','Sundries_sales_last_3y'])
)


In [0]:
# Perishables trips
# 1 year
customers_perishables_last_1y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '200000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Perishables_trips_last_1y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_Perishables_sub_cat_last_1y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Perishables_sales_last_1y'),
        
    )
)


customers_perishables_last_2y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '200000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=720),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Perishables_trips_last_2y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_Perishables_sub_cat_last_2y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Perishables_sales_last_2y'),
        
    )
)


customers_perishables_last_3y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '200000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095),filtering_date_str))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Perishables_trips_last_3y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_Perishables_sub_cat_last_3y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '200000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Perishables_sales_last_3y'),
        
    )
)

Perishables = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        customers_perishables_last_1y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_perishables_last_2y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_perishables_last_3y,
        on=['MBRSHP_SID'],
        how='left'
    )
    .fillna(0, subset=['Perishables_trips_last_1y','Perishables_trips_last_2y','Perishables_trips_last_3y','unique_Perishables_sub_cat_last_1y','unique_Perishables_sub_cat_last_2y','unique_Perishables_sub_cat_last_3y','Perishables_sales_last_1y','Perishables_sales_last_2y','Perishables_sales_last_3y'])
)


In [0]:
# Grocery trips
# 1 year
customers_grocery_last_1y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '100000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Grocery_trips_last_1y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_Grocery_sub_cat_last_1y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Grocery_sales_last_1y'),
        
    )
)


customers_grocery_last_2y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '100000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=720),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Grocery_trips_last_2y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_Grocery_sub_cat_last_2y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Grocery_sales_last_2y'),
        
    )
)


customers_grocery_last_3y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '100000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('Grocery_trips_last_3y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_Grocery_sub_cat_last_3y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '100000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('Grocery_sales_last_3y'),
        
    )
)

Grocery = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        customers_grocery_last_1y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_grocery_last_2y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_grocery_last_3y,
        on=['MBRSHP_SID'],
        how='left'
    )
    .fillna(0, subset=['Grocery_trips_last_1y','Grocery_trips_last_2y','Grocery_trips_last_3y','unique_Grocery_sub_cat_last_1y','unique_Grocery_sub_cat_last_2y','unique_Grocery_sub_cat_last_3y','Grocery_sales_last_1y','Grocery_sales_last_2y','Grocery_sales_last_3y'])
)


In [0]:
# General Merchandise trips
# 1 year
customers_GM_last_1y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '400000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('GM_trips_last_1y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_GM_sub_cat_last_1y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('GM_sales_last_1y'),
        
    )
)


customers_GM_last_2y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '400000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=720),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('GM_trips_last_2y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_GM_sub_cat_last_2y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('GM_sales_last_2y'),
        
    )
)


customers_GM_last_3y = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        purch_detail,
        on=[
            base.MBRSHP_SID == purch_detail.MBRSHP_SID              
        ],
        how='left'
    )
    .filter(f.col('MCH3_CD') == '400000000')
    .filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095),filtering_date))
    .groupBy(base.MBRSHP_SID)
    .agg(
        
        #trips
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('PURCH_HDR_ID')
                    )
        )
        .alias('GM_trips_last_3y'),
        
        #unique_categories
        f.countDistinct(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('MCH2_CD')
                    )
        )
        .alias('unique_GM_sub_cat_last_3y'),
        
        #sales
        f.sum(
                    f.when(
                        f.col('MCH3_CD') == '400000000',
                        f.col('NORMAL_PRC_AMT')
                    )
        )
        .alias('GM_sales_last_3y'),
        
    )
)

GM = (
    base.select('MBRSHP_SID').dropDuplicates().join(
        customers_GM_last_1y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_GM_last_2y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_GM_last_3y,
        on=['MBRSHP_SID'],
        how='left'
    )
    .fillna(0, subset=['GM_trips_last_1y','GM_trips_last_2y','GM_trips_last_3y','unique_GM_sub_cat_last_1y','unique_GM_sub_cat_last_2y','unique_GM_sub_cat_last_3y','GM_sales_last_1y','GM_sales_last_2y','GM_sales_last_3y'])
)


In [0]:
# subcategory trips
customer_cat = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date)),
        on = 'MBRSHP_SID',
        how='left'
    ).fillna('UNKNOWN',subset=['MCH2_DESC']).groupBy('MBRSHP_SID').pivot('MCH2_DESC').agg(f.countDistinct("PURCH_HDR_ID")))

# print(customer_cat.count())

In [0]:
# Paper coupons redeemed
coupons=['ZPAP']
customers_redeem_paper_article_cpn_last_1y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_paper_article_cpn'))
    .fillna(0, subset=['redeem_paper_article_cpn']).withColumnRenamed(
        'redeem_paper_article_cpn',
        'redeem_paper_cpn_last_1y'
    )
#    .withColumn(
#        'redeem_paper_article_cpn',
#        f.when(
#            f.col('redeem_paper_article_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)

customers_redeem_paper_article_cpn_last_2y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=720),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_paper_article_cpn'))
    .fillna(0, subset=['redeem_paper_article_cpn']).withColumnRenamed(
        'redeem_paper_article_cpn',
        'redeem_paper_cpn_last_2y'
    )
#    .withColumn(
#        'redeem_paper_article_cpn',
#        f.when(
#            f.col('redeem_paper_article_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)

customers_redeem_paper_article_cpn_last_3y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_paper_article_cpn'))
    .fillna(0, subset=['redeem_paper_article_cpn']).withColumnRenamed(
        'redeem_paper_article_cpn',
        'redeem_paper_cpn_last_3y'
    )
#    .withColumn(
#        'redeem_paper_article_cpn',
#        f.when(
#            f.col('redeem_paper_article_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)


PAPER = (base.select('MBRSHP_SID').join(customers_redeem_paper_article_cpn_last_1y,  on=['MBRSHP_SID'],
        how='left').join(
        customers_redeem_paper_article_cpn_last_2y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_redeem_paper_article_cpn_last_3y,
        on=['MBRSHP_SID'],
        how='left'
    ).fillna(0, subset=['redeem_paper_cpn_last_1y','redeem_paper_cpn_last_2y','redeem_paper_cpn_last_3y'])
)

In [0]:
# Digital coupons redeemed
coupons=['ZCOU']
customers_redeem_digital_article_cpn_last_1y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_digital_cpn'))
    .fillna(0, subset=['redeem_digital_cpn']).withColumnRenamed(
        'redeem_digital_cpn',
        'redeem_digital_cpn_last_1y'
    )
#    .withColumn(
#        'redeem_digital_cpn',
#        f.when(
#            f.col('redeem_digital_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)

customers_redeem_digital_article_cpn_last_2y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=720),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_digital_cpn'))
    .fillna(0, subset=['redeem_digital_cpn']).withColumnRenamed(
        'redeem_digital_cpn',
        'redeem_digital_cpn_last_2y'
    )
#    .withColumn(
#        'redeem_digital_cpn',
#        f.when(
#            f.col('redeem_digital_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)


customers_redeem_digital_article_cpn_last_3y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_digital_cpn'))
    .fillna(0, subset=['redeem_digital_cpn']).withColumnRenamed(
        'redeem_digital_cpn',
        'redeem_digital_cpn_last_3y'
    )
#    .withColumn(
#        'redeem_digital_cpn',
#        f.when(
#            f.col('redeem_digital_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)


DIGITAL = (base.select('MBRSHP_SID').join(customers_redeem_digital_article_cpn_last_1y,  on=['MBRSHP_SID'],
        how='left').join(
        customers_redeem_digital_article_cpn_last_2y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_redeem_digital_article_cpn_last_3y,
        on=['MBRSHP_SID'],
        how='left'
    ).fillna(0, subset=['redeem_digital_cpn_last_1y','redeem_digital_cpn_last_2y','redeem_digital_cpn_last_3y'])
)


In [0]:
# CLP coupons redeemed
coupons=['ZCLP']
customers_redeem_clp_cpn_last_1y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=365),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_clp_cpn'))
    .fillna(0, subset=['redeem_clp_cpn']).withColumnRenamed(
        'redeem_clp_cpn',
        'redeem_clp_cpn_last_1y'
    )
#    .withColumn(
#        'redeem_clp_cpn',
#        f.when(
#            f.col('redeem_clp_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)

customers_redeem_clp_cpn_last_2y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=720),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_clp_cpn'))
    .fillna(0, subset=['redeem_clp_cpn']).withColumnRenamed(
        'redeem_clp_cpn',
        'redeem_clp_cpn_last_2y'
    )
#    .withColumn(
#        'redeem_clp_cpn',
#        f.when(
#            f.col('redeem_clp_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)


customers_redeem_clp_cpn_last_3y = (
    base.select('MBRSHP_SID').join(
        purch_detail.filter(f.col('PURCH_DT').between(filtering_date - pd.DateOffset(days=1095),filtering_date)),
        on=['MBRSHP_SID'],
        how='left'
    )
    .filter(
        f.col("DISCOUNT_TYPE_CD").isin(coupons)
    )
    .groupBy(
        base.MBRSHP_SID)
    .agg(f.count('PURCH_DT').alias('redeem_clp_cpn'))
    .fillna(0, subset=['redeem_clp_cpn']).withColumnRenamed(
        'redeem_clp_cpn',
        'redeem_clp_cpn_last_3y'
    )
#    .withColumn(
#        'redeem_clp_cpn',
#        f.when(
#            f.col('redeem_clp_cpn') > 0, 1
#        )
#        .otherwise(0)
#    )
)


CLP = (base.select('MBRSHP_SID').join(customers_redeem_clp_cpn_last_1y,  on=['MBRSHP_SID'],
        how='left').join(
        customers_redeem_clp_cpn_last_2y,
        on=['MBRSHP_SID'],
        how='left'
    ).join(
        customers_redeem_clp_cpn_last_3y,
        on=['MBRSHP_SID'],
        how='left'
    ).fillna(0, subset=['redeem_clp_cpn_last_1y','redeem_clp_cpn_last_2y','redeem_clp_cpn_last_3y'])
)

In [0]:
population_base = base.join(Sundries,'mbrshp_sid', 'inner').join(Perishables,'mbrshp_sid', 'inner').join(Grocery,'mbrshp_sid', 'inner').join(GM,'mbrshp_sid', 'inner').join(customer_cat,'mbrshp_sid', 'inner').join(PAPER,'mbrshp_sid', 'inner').join(DIGITAL,'mbrshp_sid', 'inner').join(CLP,'mbrshp_sid', 'inner')


In [0]:
#cleaning the column names
import re

def clean_df_names(df):
    l = df.columns
    cols = [c.replace('&','_').replace(' ','_').strip() for c in l]
    return df.toDF(*cols)

In [0]:
population_base_cleaned = clean_df_names(population_base)

In [0]:
(
    population_base_cleaned.withColumn('RUN_DATE', f.lit(run_date_str).cast('date'))
        .withColumn('DATASET_CD', f.lit(dataset_cd))
        .write.mode('overwrite').option('replaceWhere', f"RUN_DATE = '{run_date_str}' AND DATASET_CD = '{dataset_cd}'").saveAsTable(gm_etl_output)
)